# Task 2 — Validation of Temporal Knowledge Graphs (Google Colab / Jupyter)

Этот блокнот является **естественным продолжением Task 1**.

Он сначала ищет **локальный исправленный репозиторий рядом с собой** и только если не находит его — использует `git clone` официального GitHub-репозитория.

Эксперт загружает YAML-файл, созданный в первом блокноте, после чего блокнот автоматически:
- строит **эталонный (gold) граф**, который точно воспроизводит ручную reasoning-схему из YAML;
- при желании строит **автоматический temporal KG** по тем же статьям через пайплайн репозитория;
- сохраняет обе версии графа, таблицы триплетов, comparison summary и визуализации;
- показывает эксперту **текстовое представление триплетов** и **интерактивные визуализации** для ручной валидации.

Блокнот **не патчит файлы репозитория**. Он только ставит зависимости, импортирует уже готовый API репозитория и запускает workflow.


# Пошаговый туториал

## Шаг 1. Подготовьте репозиторий
Рекомендуемый вариант — распаковать архив `top-papers-graph-fixed-v5.zip` в `/content` или рядом с ноутбуком.

## Шаг 2. Запустите ячейку «Установка и импорт»
Она:
- найдёт локальный репозиторий автоматически;
- только если локальный репозиторий не найден, клонирует официальный GitHub-репозиторий;
- установит зависимости для Task 2 notebook;
- добавит локальный `src` в `sys.path` как надёжный fallback;
- импортирует функции Task 2.

## Шаг 3. Запустите ячейку «Вспомогательные функции»
Она подготовит UI, рендеринг таблиц и графов.

## Шаг 4. Запустите ячейку «Форма Task 2»
В ней нужно:
1. загрузить YAML из Task 1;
2. выбрать, строить ли автоматический граф;
3. нажать кнопку запуска.


Обновление: после сборки bundle доступны вкладки просмотра эталонного и автоматически сгенерированного графов, а также интерфейс экспертной валидации с экспортом результатов.

In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import json, os, sys, tempfile, subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def find_repo_root() -> Path | None:
    candidates = []
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    search_bases = [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]
    patterns = (
        'top-papers-graph-fixed-v5',
        'top-papers-graph-fixed-v4',
        'top-papers-graph-fixed-v3',
        'top-papers-graph-fixed*',
        'top-papers-graph-main',
        'top-papers-graph',
    )
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {SRC_DIR}')

# 1) Лёгкий UI-стек для notebook. pandas здесь специально НЕ ставим отдельно,
#    чтобы не получить ABI-конфликт с numpy после editable-install репозитория.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ipywidgets', 'pyyaml', 'requests',
    'panel', 'hvplot', 'holoviews', 'bokeh', 'pyvis', 'jupyter_bokeh'
])

# 2) Ставим extras репозитория.
#    Сначала пробуем единый extra из исправленного архива, затем fallback для GitHub upstream.
try:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', f'{REPO_DIR}[task2_notebook]'])
except Exception:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', f'{REPO_DIR}[temporal,mm]'])

# 3) В Colab часто остаётся предустановленный pandas, собранный под другой ABI NumPy.
#    Принудительно выравниваем совместимую пару после editable-install.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir',
    'numpy>=1.26.4,<2', 'pandas>=2.2.2,<2.4'
])

# 4) Даже если editable-install не прописал import hook, ноутбук всё равно должен работать.
#    Поэтому всегда добавляем локальный src в sys.path как надёжный fallback.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# На случай, если kernel уже видел старые модули
for mod in list(sys.modules):
    if mod == 'numpy' or mod.startswith('numpy.') or mod == 'pandas' or mod.startswith('pandas.'):
        del sys.modules[mod]

import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, HTML, clear_output

try:
    import panel as pn
    pn.extension()
except Exception:
    pn = None

try:
    import hvplot  # noqa: F401
    import hvplot.networkx  # noqa: F401
except Exception:
    pass

try:
    from scireason.task2_validation import (
        build_task2_validation_bundle,
        load_task1_yaml,
        make_hvplot_payload,
    )
except Exception:
    from scireason.pipeline.task2_validation import prepare_task2_validation_bundle as _prepare_task2_validation_bundle

    def build_task2_validation_bundle(*args, **kwargs):
        bundle_dir = _prepare_task2_validation_bundle(*args, **kwargs)
        class _Result:
            def __init__(self, bundle_dir):
                self.bundle_dir = Path(bundle_dir)
                self.manifest_path = self.bundle_dir / 'task2_notebook_manifest.json'
        return _Result(bundle_dir)

    def load_task1_yaml(path):
        import yaml
        return yaml.safe_load(Path(path).read_text(encoding='utf-8'))

    def make_hvplot_payload(payload):
        import networkx as nx
        G = nx.DiGraph()
        for node in payload.get('nodes', []) or []:
            node_id = node.get('id') or node.get('term') or node.get('label')
            if node_id is not None:
                G.add_node(str(node_id), **dict(node))
        for edge in payload.get('edges', []) or []:
            src = edge.get('source')
            tgt = edge.get('target') or edge.get('object')
            if src is not None and tgt is not None:
                G.add_edge(str(src), str(tgt), **dict(edge))
        try:
            import hvplot.networkx as hvnx  # noqa: F401
            pos = nx.spring_layout(G, seed=7)
            plot = hvnx.draw(G, pos, with_labels=False, width=950, height=650)
            return G, plot
        except Exception:
            return G, None

print('Готово.')
print('Repo dir:', REPO_DIR)
print('Src dir:', SRC_DIR)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


In [ ]:

# @title
# Вспомогательные функции (не нужно редактировать)
from pathlib import Path
from datetime import datetime
import ast
import json
import zipfile
import yaml


def _save_uploaded_yaml(upload_value, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)

    missing_msg = (
        'YAML-файл не загружен. Нажмите «Загрузить YAML», выберите файл '
        'с расширением .yaml или .yml и запустите кнопку ещё раз.'
    )

    if not upload_value:
        raise ValueError(missing_msg)

    if isinstance(upload_value, dict):
        if not upload_value:
            raise ValueError(missing_msg)
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            raise ValueError('Загруженный YAML пустой или повреждён. Загрузите файл заново.')
        path = out_dir / (name or 'task1.yaml')
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path

    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        name = meta.get('name') or 'task1.yaml'
        content = meta.get('content')
        if content is None:
            raise ValueError('Загруженный YAML пустой или повреждён. Загрузите файл заново.')
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path

    raise ValueError(missing_msg)


def _read_uploaded_yaml_bytes(upload_value):
    if not upload_value:
        return None, None

    if isinstance(upload_value, dict) and upload_value:
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        return name or 'task1.yaml', content

    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        if isinstance(meta, dict):
            return meta.get('name') or 'task1.yaml', meta.get('content')

    return None, None


def _format_size(num_bytes):
    if num_bytes is None:
        return None
    num = float(num_bytes)
    units = ['B', 'KB', 'MB', 'GB']
    for unit in units:
        if num < 1024 or unit == units[-1]:
            if unit == 'B':
                return f'{int(num)} {unit}'
            return f'{num:.1f} {unit}'
        num /= 1024.0


def _inspect_uploaded_yaml(upload_value):
    name, content = _read_uploaded_yaml_bytes(upload_value)
    if not name or content is None:
        return {
            'state': 'missing',
            'name': None,
            'size_bytes': None,
            'size_label': None,
            'submission_id': None,
            'topic': None,
            'message': 'YAML не загружен',
        }

    try:
        raw = content if isinstance(content, (bytes, bytearray)) else bytes(content)
    except Exception:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': None,
            'size_label': None,
            'submission_id': None,
            'topic': None,
            'message': 'Не удалось прочитать файл',
        }

    if not raw:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': 0,
            'size_label': '0 B',
            'submission_id': None,
            'topic': None,
            'message': 'Файл пустой',
        }

    size_label = _format_size(len(raw))

    try:
        text = raw.decode('utf-8')
    except UnicodeDecodeError:
        try:
            text = raw.decode('utf-8-sig')
        except UnicodeDecodeError:
            text = raw.decode('latin-1')

    try:
        parsed = yaml.safe_load(text) or {}
    except Exception as e:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': len(raw),
            'size_label': size_label,
            'submission_id': None,
            'topic': None,
            'message': f'YAML не распознан: {type(e).__name__}',
        }

    if not isinstance(parsed, dict):
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': len(raw),
            'size_label': size_label,
            'submission_id': None,
            'topic': None,
            'message': 'В YAML ожидается объект верхнего уровня',
        }

    submission_id = parsed.get('submission_id')
    topic = parsed.get('topic')
    return {
        'state': 'valid',
        'name': name,
        'size_bytes': len(raw),
        'size_label': size_label,
        'submission_id': submission_id,
        'topic': topic,
        'message': 'YAML загружен',
    }


def _display_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    lines = [
        f"- **topic**: {manifest.get('topic')}",
        f"- **bundle_dir**: `{manifest.get('bundle_dir')}`",
        f"- **gold_graph**: `{manifest.get('gold_graph')}`",
        f"- **gold_triplets_csv**: `{manifest.get('gold_triplets_csv')}`",
    ]
    if manifest.get('auto_run_dir'):
        lines += [
            f"- **auto_run_dir**: `{manifest.get('auto_run_dir')}`",
            f"- **auto_graph_json**: `{manifest.get('auto_graph_json')}`",
            f"- **auto_triplets_csv**: `{manifest.get('auto_triplets_csv')}`",
        ]
    if manifest.get('comparison_summary'):
        lines.append(f"- **comparison_summary**: `{manifest.get('comparison_summary')}`")
    if manifest.get('reference_scout'):
        lines.append(f"- **reference_scout**: `{manifest.get('reference_scout')}`")
    display(Markdown("## Артефакты Task 2\n" + "\n".join(lines)))
    return manifest


def _safe_json_load(path):
    p = Path(path)
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding='utf-8'))


def _parse_maybe_object(value):
    if isinstance(value, (dict, list)):
        return value
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return text


def _format_evidence_short(value):
    obj = _parse_maybe_object(value)
    if isinstance(obj, dict):
        parts = []
        paper_id = obj.get('paper_id') or obj.get('id')
        snippet = obj.get('snippet_or_summary') or obj.get('snippet') or obj.get('summary') or obj.get('quote')
        page = obj.get('page')
        if paper_id:
            parts.append(f"paper: {paper_id}")
        if page not in (None, '', 'nan'):
            parts.append(f"page: {page}")
        if snippet:
            parts.append(str(snippet)[:220])
        return " | ".join(parts) if parts else json.dumps(obj, ensure_ascii=False)
    if isinstance(obj, list):
        return "; ".join(str(x) for x in obj[:5])
    return '' if obj is None else str(obj)


def _read_triplets_table(path):
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    if p.suffix.lower() == '.json':
        payload = _safe_json_load(p)
        if isinstance(payload, list):
            return pd.DataFrame(payload)
        if isinstance(payload, dict):
            for key in ('assertions', 'rows', 'triplets', 'edges'):
                if isinstance(payload.get(key), list):
                    return pd.DataFrame(payload[key])
            return pd.DataFrame([payload])
    return pd.read_csv(p)


def _pick_first(row, keys, default=''):
    for key in keys:
        if key in row:
            value = row.get(key)
            if value is None:
                continue
            if isinstance(value, float) and pd.isna(value):
                continue
            text = str(value)
            if text.strip() == '':
                continue
            return value
    return default


def _normalize_assertions(df, graph_kind):
    if df is None or df.empty:
        return pd.DataFrame(columns=[
            'graph_kind', 'assertion_id', 'edge_uid', 'subject', 'predicate', 'object',
            'start_date', 'end_date', 'valid_from', 'valid_to', 'time_source',
            'time_interval', 'evidence_text', 'papers_text', 'score', 'mean_confidence',
            'verdict', 'rationale', 'time_source_note', 'raw_record_json'
        ])

    rows = []
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        row_dict = row.to_dict()
        assertion_id = str(_pick_first(row_dict, ['assertion_id', 'id'], f'{graph_kind}-{idx:05d}'))
        subject = str(_pick_first(row_dict, ['subject', 'source', 'src', 'from'], ''))
        predicate = str(_pick_first(row_dict, ['predicate', 'relation', 'label', 'type'], ''))
        object_ = str(_pick_first(row_dict, ['object', 'target', 'dst', 'to'], ''))
        start_date = str(_pick_first(row_dict, ['start_date', 'start', 'begin', 'year_start'], ''))
        end_date = str(_pick_first(row_dict, ['end_date', 'end', 'finish', 'year_end'], ''))
        valid_from = str(_pick_first(row_dict, ['valid_from', 'vf'], start_date))
        valid_to = str(_pick_first(row_dict, ['valid_to', 'vt'], end_date or '+inf'))
        time_source = str(_pick_first(row_dict, ['time_source', 'temporal_source'], ''))
        time_interval = str(_pick_first(row_dict, ['time_interval'], ''))
        papers_text = _parse_maybe_object(_pick_first(row_dict, ['papers'], []))
        if isinstance(papers_text, list):
            papers_text = ', '.join(str(x) for x in papers_text)
        evidence_text = _format_evidence_short(_pick_first(row_dict, ['evidence', 'snippet', 'quote', 'summary'], ''))
        rows.append({
            'graph_kind': graph_kind,
            'assertion_id': assertion_id,
            'edge_uid': f'{graph_kind}:{assertion_id}',
            'subject': subject,
            'predicate': predicate,
            'object': object_,
            'start_date': start_date,
            'end_date': end_date,
            'valid_from': valid_from,
            'valid_to': valid_to,
            'time_source': time_source,
            'time_interval': time_interval,
            'evidence_text': evidence_text,
            'papers_text': papers_text or '',
            'score': _pick_first(row_dict, ['score'], ''),
            'mean_confidence': _pick_first(row_dict, ['mean_confidence'], ''),
            'verdict': str(_pick_first(row_dict, ['verdict'], '')),
            'rationale': str(_pick_first(row_dict, ['rationale'], '')),
            'time_source_note': str(_pick_first(row_dict, ['time_source_note'], '')),
            'raw_record_json': json.dumps(row_dict, ensure_ascii=False),
        })
    return pd.DataFrame(rows)


def _build_assertions_panel(assertions_df, title):
    search = W.Text(
        value='',
        description='Поиск',
        placeholder='subject / predicate / object / evidence',
        layout=W.Layout(width='70%')
    )
    rows_slider = W.IntSlider(value=20, min=5, max=100, step=5, description='Строк')
    output = W.Output()

    def render(*_):
        with output:
            clear_output()
            display(Markdown(f"### {title} — assertions"))
            df = assertions_df.copy()
            query = search.value.strip().lower()
            if query:
                mask = (
                    df['subject'].astype(str).str.lower().str.contains(query, na=False) |
                    df['predicate'].astype(str).str.lower().str.contains(query, na=False) |
                    df['object'].astype(str).str.lower().str.contains(query, na=False) |
                    df['evidence_text'].astype(str).str.lower().str.contains(query, na=False)
                )
                df = df[mask]
            display(Markdown(f"Найдено строк: **{len(df)}**"))
            show_cols = [
                'assertion_id', 'subject', 'predicate', 'object',
                'start_date', 'end_date', 'valid_from', 'valid_to',
                'time_source', 'score', 'mean_confidence', 'evidence_text'
            ]
            existing = [c for c in show_cols if c in df.columns]
            display(df[existing].head(rows_slider.value))

    search.observe(render, names='value')
    rows_slider.observe(render, names='value')
    render()
    return W.VBox([W.HBox([search, rows_slider]), output])


def _build_visualization_panel(graph_json_path, html_path, title):
    output = W.Output()
    refresh_btn = W.Button(description='Обновить визуализацию', button_style='info')

    def render(*_):
        with output:
            clear_output()
            display(Markdown(f"### {title} — визуализация"))
            gj = Path(graph_json_path)
            hp = Path(html_path)
            if gj.exists():
                try:
                    payload = json.loads(gj.read_text(encoding='utf-8'))
                    try:
                        _, plot = make_hvplot_payload(payload)
                        if plot is not None:
                            display(plot)
                    except Exception as e:
                        display(Markdown(f'hvPlot не удалось построить: `{e}`'))
                    display(Markdown(
                        f"JSON: `{gj}`  \n"
                        f"Nodes: **{len(payload.get('nodes', []) or [])}**, edges: **{len(payload.get('edges', []) or [])}**"
                    ))
                except Exception as e:
                    display(Markdown(f'Не удалось открыть JSON графа: `{e}`'))
            else:
                display(Markdown('JSON графа не найден.'))
            if hp.exists():
                try:
                    html = hp.read_text(encoding='utf-8')
                    display(HTML(html))
                except Exception:
                    display(Markdown(f'HTML-файл графа: `{hp}`'))
            else:
                display(Markdown('HTML-визуализация не найдена.'))

    refresh_btn.on_click(render)
    render()
    return W.VBox([refresh_btn, output])


def _build_graph_view(manifest):
    gold_df = _normalize_assertions(_read_triplets_table(manifest['gold_triplets_csv']), 'gold')
    gold_tabs = W.Tab(children=[
        _build_assertions_panel(gold_df, 'Эталонный граф'),
        _build_visualization_panel(manifest['gold_graph'], manifest['gold_graph_html'], 'Эталонный граф'),
    ])
    gold_tabs.set_title(0, 'Assertions')
    gold_tabs.set_title(1, 'Визуализация')

    graph_tabs_children = [gold_tabs]
    graph_titles = ['Эталонный граф']

    if manifest.get('auto_triplets_csv'):
        auto_df = _normalize_assertions(_read_triplets_table(manifest['auto_triplets_csv']), 'auto')
        auto_tabs = W.Tab(children=[
            _build_assertions_panel(auto_df, 'Авто-граф'),
            _build_visualization_panel(manifest['auto_graph_json'], manifest['auto_graph_html'], 'Авто-граф'),
        ])
        auto_tabs.set_title(0, 'Assertions')
        auto_tabs.set_title(1, 'Визуализация')
        graph_tabs_children.append(auto_tabs)
        graph_titles.append('Авто-граф')

    graph_tabs = W.Tab(children=graph_tabs_children)
    for idx, title in enumerate(graph_titles):
        graph_tabs.set_title(idx, title)
    return graph_tabs


def _show_reference_scout_panel(path):
    p = Path(path)
    output = W.Output()
    with output:
        if not p.exists():
            display(Markdown('Reference scout не найден.'))
        else:
            scout = json.loads(p.read_text(encoding='utf-8'))
            rows = []
            if isinstance(scout, dict):
                for hit in scout.get('hits', []):
                    for res in hit.get('results', []):
                        rows.append({
                            'query': hit.get('query'),
                            'id': res.get('id'),
                            'title': res.get('title'),
                            'year': res.get('year'),
                            'url': res.get('url'),
                        })
            elif isinstance(scout, list):
                for res in scout:
                    if not isinstance(res, dict):
                        continue
                    rows.append({
                        'query': ', '.join(res.get('trigger_queries') or []),
                        'id': res.get('paper_id') or res.get('id'),
                        'title': res.get('title'),
                        'year': res.get('year'),
                        'url': res.get('url'),
                        'score': res.get('score'),
                    })
            display(Markdown('### Reference scout'))
            if rows:
                display(pd.DataFrame(rows).head(100))
            else:
                display(Markdown('Кандидаты не найдены.'))
    return output


def _build_summary_panel(manifest):
    output = W.Output()
    with output:
        if manifest.get('comparison_summary'):
            cmp = json.loads(Path(manifest['comparison_summary']).read_text(encoding='utf-8'))
            display(Markdown('### Comparison summary'))
            display(pd.DataFrame([cmp]))
        else:
            display(Markdown('Comparison summary не найден.'))
    return output


def _default_review_state(row):
    return {
        'verdict': str(row.get('verdict') or ''),
        'rationale': str(row.get('rationale') or ''),
        'time_source_note': str(row.get('time_source_note') or ''),
        'corrected_start_date': '',
        'corrected_end_date': '',
        'corrected_valid_from': '',
        'corrected_valid_to': '',
        'corrected_time_source': '',
        'correction_comment': '',
    }


def _build_validation_workspace(manifest, task1_doc):
    frames = []
    gold_df = _normalize_assertions(_read_triplets_table(manifest['gold_triplets_csv']), 'gold')
    if not gold_df.empty:
        frames.append(gold_df)
    if manifest.get('auto_triplets_csv'):
        auto_df = _normalize_assertions(_read_triplets_table(manifest['auto_triplets_csv']), 'auto')
        if not auto_df.empty:
            frames.append(auto_df)

    if not frames:
        return W.HTML('<b>Нет данных для валидации.</b>')

    combined = pd.concat(frames, ignore_index=True)
    review_state = {row['edge_uid']: _default_review_state(row) for _, row in combined.iterrows()}

    reviewer_default = ''
    expert = task1_doc.get('expert') if isinstance(task1_doc.get('expert'), dict) else {}
    if expert:
        reviewer_default = str(expert.get('latin_slug') or expert.get('full_name') or '')
    reviewer_id = W.Text(value=reviewer_default, description='Reviewer', layout=W.Layout(width='55%'))
    graph_filter = W.Dropdown(
        options=[('Все графы', 'all'), ('Эталонный', 'gold'), ('Авто', 'auto')],
        value='all',
        description='Граф'
    )
    verdict_filter = W.Dropdown(
        options=[
            ('Все', 'all'),
            ('Без решения', 'pending'),
            ('accepted', 'accepted'),
            ('rejected', 'rejected'),
            ('uncertain', 'uncertain'),
            ('needs_time_fix', 'needs_time_fix'),
        ],
        value='pending',
        description='Фильтр'
    )
    search = W.Text(
        value='',
        description='Поиск',
        placeholder='по subject / predicate / object / evidence',
        layout=W.Layout(width='60%')
    )
    page_size = W.IntSlider(value=5, min=1, max=10, step=1, description='На стр.')
    page = W.IntSlider(value=1, min=1, max=1, step=1, description='Стр.')
    summary_html = W.HTML()
    editor_output = W.Output(layout=W.Layout(border='1px solid #e2e8f0', padding='8px'))
    export_status = W.HTML()
    export_btn = W.Button(description='Сохранить файлы валидации', button_style='success')
    refresh_btn = W.Button(description='Обновить список', button_style='info')
    download_btn = W.Button(description='Скачать ZIP', button_style='warning')
    download_btn.disabled = True

    def filtered_df():
        df = combined.copy()
        if graph_filter.value != 'all':
            df = df[df['graph_kind'] == graph_filter.value]

        query = search.value.strip().lower()
        if query:
            mask = (
                df['subject'].astype(str).str.lower().str.contains(query, na=False) |
                df['predicate'].astype(str).str.lower().str.contains(query, na=False) |
                df['object'].astype(str).str.lower().str.contains(query, na=False) |
                df['evidence_text'].astype(str).str.lower().str.contains(query, na=False)
            )
            df = df[mask]

        if verdict_filter.value != 'all':
            if verdict_filter.value == 'pending':
                mask = [not review_state[row['edge_uid']]['verdict'] for _, row in df.iterrows()]
                df = df[pd.Series(mask, index=df.index)]
            else:
                mask = [review_state[row['edge_uid']]['verdict'] == verdict_filter.value for _, row in df.iterrows()]
                df = df[pd.Series(mask, index=df.index)]

        return df.reset_index(drop=True)

    def make_observer(edge_uid, key):
        def _observer(change):
            review_state[edge_uid][key] = change['new']
            update_summary()
        return _observer

    def update_summary(*_):
        total = len(combined)
        decided = sum(1 for st in review_state.values() if st['verdict'])
        corrected = sum(
            1 for st in review_state.values()
            if any(st.get(k) for k in (
                'corrected_start_date', 'corrected_end_date',
                'corrected_valid_from', 'corrected_valid_to',
                'corrected_time_source', 'correction_comment'
            ))
        )
        pending = total - decided
        summary_html.value = (
            '<div style="padding:6px 0;color:#334155">'
            f'<b>Всего рёбер:</b> {total} &nbsp; '
            f'<b>С оценкой:</b> {decided} &nbsp; '
            f'<b>Без оценки:</b> {pending} &nbsp; '
            f'<b>С временными правками:</b> {corrected}'
            '</div>'
        )

    def render_editors(*_):
        df = filtered_df()
        total_pages = max(1, int((len(df) + page_size.value - 1) / page_size.value))
        page.max = total_pages
        if page.value > total_pages:
            page.value = total_pages
        start = (page.value - 1) * page_size.value
        visible = df.iloc[start:start + page_size.value]

        with editor_output:
            clear_output()
            display(Markdown(
                '### Валидация рёбер\n'
                'Для каждого ребра выберите verdict и при необходимости внесите корректировки временных полей. '
                'После этого нажмите **«Сохранить файлы валидации»**.'
            ))
            if visible.empty:
                display(Markdown('По текущим фильтрам строки не найдены.'))
                return

            items = []
            for _, row in visible.iterrows():
                state = review_state[row['edge_uid']]

                header = W.HTML(
                    value=(
                        f"<div style='font-size:13px'><b>[{row['graph_kind']}]</b> "
                        f"{_escape_html(row['subject'])} "
                        f"<span style='color:#0f766e'>— { _escape_html(row['predicate']) } →</span> "
                        f"{_escape_html(row['object'])}</div>"
                    )
                )
                meta = W.HTML(
                    value=(
                        "<div style='font-size:12px;color:#475569'>"
                        f"assertion_id: <code>{_escape_html(row['assertion_id'])}</code> · "
                        f"start/end: <code>{_escape_html(row['start_date'])}</code> / <code>{_escape_html(row['end_date'])}</code> · "
                        f"valid: <code>{_escape_html(row['valid_from'])}</code> / <code>{_escape_html(row['valid_to'])}</code>"
                        "</div>"
                    )
                )
                evidence = W.HTML(
                    value=f"<div style='font-size:12px;color:#475569'><b>Evidence:</b> {_escape_html(row['evidence_text'] or '—')}</div>"
                )

                verdict = W.Dropdown(
                    options=[
                        ('', ''),
                        ('accepted', 'accepted'),
                        ('rejected', 'rejected'),
                        ('uncertain', 'uncertain'),
                        ('needs_time_fix', 'needs_time_fix'),
                    ],
                    value=state['verdict'],
                    description='Verdict',
                    layout=W.Layout(width='260px')
                )
                rationale = W.Textarea(
                    value=state['rationale'],
                    description='Почему',
                    placeholder='Комментарий эксперта по ребру',
                    layout=W.Layout(width='100%', height='70px')
                )
                time_source_note = W.Text(
                    value=state['time_source_note'],
                    description='Источник времени',
                    placeholder='например: подтверждено по full-text / figure / table'
                )

                corrected_start = W.Text(value=state['corrected_start_date'], description='start_date')
                corrected_end = W.Text(value=state['corrected_end_date'], description='end_date')
                corrected_vf = W.Text(value=state['corrected_valid_from'], description='valid_from')
                corrected_vt = W.Text(value=state['corrected_valid_to'], description='valid_to')
                corrected_ts = W.Text(value=state['corrected_time_source'], description='time_source')
                correction_comment = W.Textarea(
                    value=state['correction_comment'],
                    description='Правка',
                    placeholder='Что именно нужно исправить во времени',
                    layout=W.Layout(width='100%', height='70px')
                )

                for widget, key in [
                    (verdict, 'verdict'),
                    (rationale, 'rationale'),
                    (time_source_note, 'time_source_note'),
                    (corrected_start, 'corrected_start_date'),
                    (corrected_end, 'corrected_end_date'),
                    (corrected_vf, 'corrected_valid_from'),
                    (corrected_vt, 'corrected_valid_to'),
                    (corrected_ts, 'corrected_time_source'),
                    (correction_comment, 'correction_comment'),
                ]:
                    widget.observe(make_observer(row['edge_uid'], key), names='value')

                reset_btn = W.Button(description='Очистить правки', button_style='')
                def _make_reset(edge_uid, widgets):
                    def _reset(_):
                        review_state[edge_uid] = _default_review_state(row)
                        defaults = review_state[edge_uid]
                        for w, key in widgets:
                            w.value = defaults[key]
                        update_summary()
                    return _reset
                reset_btn.on_click(_make_reset(row['edge_uid'], [
                    (verdict, 'verdict'),
                    (rationale, 'rationale'),
                    (time_source_note, 'time_source_note'),
                    (corrected_start, 'corrected_start_date'),
                    (corrected_end, 'corrected_end_date'),
                    (corrected_vf, 'corrected_valid_from'),
                    (corrected_vt, 'corrected_valid_to'),
                    (corrected_ts, 'corrected_time_source'),
                    (correction_comment, 'correction_comment'),
                ]))

                box = W.VBox([
                    header,
                    meta,
                    evidence,
                    W.HBox([verdict, reset_btn]),
                    rationale,
                    time_source_note,
                    W.HTML('<b>Коррекция временных параметров</b>'),
                    W.HBox([corrected_start, corrected_end]),
                    W.HBox([corrected_vf, corrected_vt]),
                    corrected_ts,
                    correction_comment,
                ])
                items.append(box)

            accordion = W.Accordion(children=items)
            for i, (_, row) in enumerate(visible.iterrows()):
                accordion.set_title(i, f"[{row['graph_kind']}] {row['subject']} — {row['predicate']} → {row['object']}"[:120])
            display(accordion)

    def _build_export_frames():
        review_rows = []
        correction_rows = []
        for _, row in combined.iterrows():
            state = review_state[row['edge_uid']]
            merged = row.to_dict()
            merged.update({
                'reviewer_id': reviewer_id.value.strip(),
                'review_timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
                'expert_verdict': state['verdict'],
                'expert_rationale': state['rationale'],
                'expert_time_source_note': state['time_source_note'],
                'corrected_start_date': state['corrected_start_date'],
                'corrected_end_date': state['corrected_end_date'],
                'corrected_valid_from': state['corrected_valid_from'],
                'corrected_valid_to': state['corrected_valid_to'],
                'corrected_time_source': state['corrected_time_source'],
                'correction_comment': state['correction_comment'],
            })
            review_rows.append(merged)

            has_correction = any(state.get(k) for k in [
                'corrected_start_date', 'corrected_end_date', 'corrected_valid_from',
                'corrected_valid_to', 'corrected_time_source', 'correction_comment'
            ])
            if has_correction:
                correction_rows.append({
                    'edge_uid': row['edge_uid'],
                    'graph_kind': row['graph_kind'],
                    'assertion_id': row['assertion_id'],
                    'subject': row['subject'],
                    'predicate': row['predicate'],
                    'object': row['object'],
                    'original_start_date': row['start_date'],
                    'original_end_date': row['end_date'],
                    'original_valid_from': row['valid_from'],
                    'original_valid_to': row['valid_to'],
                    'corrected_start_date': state['corrected_start_date'],
                    'corrected_end_date': state['corrected_end_date'],
                    'corrected_valid_from': state['corrected_valid_from'],
                    'corrected_valid_to': state['corrected_valid_to'],
                    'corrected_time_source': state['corrected_time_source'],
                    'comment': state['correction_comment'],
                    'reviewer_id': reviewer_id.value.strip(),
                })

        return pd.DataFrame(review_rows), pd.DataFrame(correction_rows)

    last_export = {'zip_path': None}

    def export_reviews(_):
        review_df, corrections_df = _build_export_frames()
        bundle_dir = Path(manifest['bundle_dir'])
        export_dir = bundle_dir / 'expert_validation'
        export_dir.mkdir(parents=True, exist_ok=True)

        review_csv = export_dir / 'edge_reviews.csv'
        review_json = export_dir / 'edge_reviews.json'
        corrections_csv = export_dir / 'temporal_corrections.csv'
        corrections_json = export_dir / 'temporal_corrections.json'
        summary_json = export_dir / 'validation_summary.json'
        zip_path = export_dir / 'expert_validation_bundle.zip'

        review_df.to_csv(review_csv, index=False)
        corrections_df.to_csv(corrections_csv, index=False)

        review_payload = {
            'artifact_version': 4,
            'domain': str(task1_doc.get('domain') or ''),
            'topic': str(task1_doc.get('topic') or ''),
            'trajectory_submission_id': str(task1_doc.get('submission_id') or ''),
            'reviewer_id': reviewer_id.value.strip(),
            'timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
            'assertions': review_df.to_dict(orient='records'),
        }
        corrections_payload = {
            'artifact_version': 3,
            'domain': str(task1_doc.get('domain') or ''),
            'paper_id': '',
            'reviewer_id': reviewer_id.value.strip(),
            'trajectory_submission_id': str(task1_doc.get('submission_id') or ''),
            'corrections': corrections_df.to_dict(orient='records'),
        }
        summary_payload = {
            'total_edges': int(len(review_df)),
            'decided_edges': int((review_df['expert_verdict'].fillna('') != '').sum()) if 'expert_verdict' in review_df else 0,
            'accepted_edges': int((review_df['expert_verdict'].fillna('') == 'accepted').sum()) if 'expert_verdict' in review_df else 0,
            'rejected_edges': int((review_df['expert_verdict'].fillna('') == 'rejected').sum()) if 'expert_verdict' in review_df else 0,
            'uncertain_edges': int((review_df['expert_verdict'].fillna('') == 'uncertain').sum()) if 'expert_verdict' in review_df else 0,
            'needs_time_fix_edges': int((review_df['expert_verdict'].fillna('') == 'needs_time_fix').sum()) if 'expert_verdict' in review_df else 0,
            'temporal_corrections': int(len(corrections_df)),
            'export_dir': str(export_dir),
        }

        review_json.write_text(json.dumps(review_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        corrections_json.write_text(json.dumps(corrections_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

        with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            for p in [review_csv, review_json, corrections_csv, corrections_json, summary_json]:
                zf.write(p, arcname=p.name)

        last_export['zip_path'] = zip_path
        download_btn.disabled = False
        export_status.value = (
            '<div style="color:#166534"><b>Файлы валидации сохранены.</b><br>'
            f'CSV с оценкой рёбер: <code>{review_csv}</code><br>'
            f'CSV с временными правками: <code>{corrections_csv}</code><br>'
            f'ZIP для выгрузки: <code>{zip_path}</code>'
            '</div>'
        )

    def download_export(_):
        zip_path = last_export.get('zip_path')
        if not zip_path:
            export_status.value = '<span style="color:#b45309">Сначала сохраните файлы валидации.</span>'
            return
        try:
            from google.colab import files as colab_files
            colab_files.download(str(zip_path))
        except Exception:
            export_status.value += (
                '<div style="color:#475569">Автоскачивание недоступно в этом окружении. '
                f'Заберите ZIP по пути: <code>{zip_path}</code></div>'
            )

    for widget in [graph_filter, verdict_filter, search, page_size]:
        widget.observe(render_editors, names='value')
    page.observe(render_editors, names='value')
    refresh_btn.on_click(render_editors)
    export_btn.on_click(export_reviews)
    download_btn.on_click(download_export)

    update_summary()
    render_editors()

    return W.VBox([
        W.HTML(
            '<div style="margin:6px 0 10px 0;color:#475569">'
            'Здесь эксперт оценивает каждое ребро и при необходимости корректирует его временные параметры. '
            'На выходе ноутбук сформирует готовые CSV/JSON и ZIP с результатами.'
            '</div>'
        ),
        W.HBox([reviewer_id, summary_html]),
        W.HBox([graph_filter, verdict_filter]),
        W.HBox([search, page_size, page]),
        W.HBox([refresh_btn, export_btn, download_btn]),
        export_status,
        editor_output,
    ])


def _build_review_workspace(manifest, task1_doc):
    children = [
        _build_graph_view(manifest),
        _build_validation_workspace(manifest, task1_doc),
    ]
    titles = ['Просмотр графов', 'Валидация эксперта']

    if manifest.get('comparison_summary'):
        children.append(_build_summary_panel(manifest))
        titles.append('Сводка')
    if manifest.get('reference_scout'):
        children.append(_show_reference_scout_panel(manifest['reference_scout']))
        titles.append('Reference scout')

    tabs = W.Tab(children=children)
    for idx, title in enumerate(titles):
        tabs.set_title(idx, title)
    return tabs


In [ ]:

# @title
# Форма Task 2 (запустите ячейку и заполните поля)

yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
yaml_indicator = W.HTML()
out_dir = W.Text(value='runs/task2_validation', description='Out dir', layout=W.Layout(width='100%'))
run_auto = W.Checkbox(value=True, description='Строить автоматический temporal KG')
reference_scout = W.Checkbox(value=True, description='Сгенерировать reference scout')
multimodal = W.Checkbox(value=True, description='Использовать multimodal pipeline')
run_btn = W.Button(description='Построить Task 2 bundle', button_style='success')
status = W.HTML()
out = W.Output()


def _yaml_indicator_chip(color_text, color_bg, color_border, dot_color, label_html):
    return (
        '<span style="display:inline-flex;align-items:center;gap:6px;'
        f'font-size:12px;color:{color_text};background:{color_bg};border:1px solid {color_border};'
        'border-radius:999px;padding:2px 8px;max-width:100%;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
        f'<span style="width:8px;height:8px;border-radius:999px;background:{dot_color};display:inline-block;flex:0 0 auto"></span>'
        f'<span style="overflow:hidden;text-overflow:ellipsis">{label_html}</span>'
        '</span>'
    )


def _escape_html(text):
    if text is None:
        return ''
    return (
        str(text)
        .replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
    )


def _update_yaml_indicator(*_):
    info = _inspect_uploaded_yaml(yaml_upload.value)

    if info['state'] == 'valid':
        primary = f"<b>YAML загружен</b>: {_escape_html(info['name'])}"
        if info.get('submission_id'):
            secondary = f" · submission_id: <code>{_escape_html(info['submission_id'])}</code>"
        else:
            secondary = f" · размер: {_escape_html(info.get('size_label') or '—')}"
        yaml_indicator.value = _yaml_indicator_chip(
            '#166534', '#f0fdf4', '#bbf7d0', '#22c55e', primary + secondary
        )
        return

    if info['state'] == 'invalid':
        primary = f"<b>Проверьте YAML</b>: {_escape_html(info['name'] or 'файл')}"
        details = info.get('message') or 'Не удалось быстро проверить файл'
        extra = f" · {_escape_html(details)}"
        if info.get('size_label'):
            extra += f" · размер: {_escape_html(info['size_label'])}"
        yaml_indicator.value = _yaml_indicator_chip(
            '#92400e', '#fffbeb', '#fde68a', '#f59e0b', primary + extra
        )
        return

    yaml_indicator.value = _yaml_indicator_chip(
        '#64748b', '#f8fafc', '#e2e8f0', '#cbd5e1', 'YAML не загружен'
    )


yaml_upload.observe(_update_yaml_indicator, names='value')
_update_yaml_indicator()


def _on_run(_):
    with out:
        clear_output()
        try:
            info = _inspect_uploaded_yaml(yaml_upload.value)
            if info['state'] == 'missing':
                status.value = (
                    '<span style="color:#b45309"><b>Нужен YAML-файл.</b> '
                    'Сначала нажмите «Загрузить YAML» и выберите файл .yaml/.yml, '
                    'после этого повторно нажмите «Построить Task 2 bundle».</span>'
                )
                display(Markdown(
                    '### Нужно загрузить YAML\n'
                    '1. Нажмите **«Загрузить YAML»**.\n'
                    '2. Выберите файл **Task 1** с расширением `.yaml` или `.yml`.\n'
                    '3. Дождитесь зелёного индикатора **«YAML загружен»**.\n'
                    '4. Если доступен `submission_id`, он сразу появится рядом с названием файла.\n'
                    '5. Снова нажмите **«Построить Task 2 bundle»**.'
                ))
                return

            if info['state'] == 'invalid':
                status.value = (
                    '<span style="color:#b45309"><b>Нужно проверить YAML.</b> '
                    'Файл загружен, но быстрая проверка показала проблему. '
                    'Исправьте YAML и загрузите его заново.</span>'
                )
                display(Markdown(
                    '### YAML загружен, но не прошёл быструю проверку\n'
                    f'- **Файл:** `{info.get("name") or "unknown"}`\n'
                    f'- **Проблема:** {info.get("message") or "неизвестная ошибка"}\n'
                    f'- **Размер:** {info.get("size_label") or "—"}\n\n'
                    'После исправления снова загрузите файл и дождитесь зелёного индикатора.'
                ))
                return

            status.value = '<b>Запуск...</b>'
            workdir = Path(tempfile.mkdtemp(prefix='task2_notebook_'))
            trajectory_path = _save_uploaded_yaml(yaml_upload.value, workdir)
            task1 = load_task1_yaml(trajectory_path)
            display(Markdown(
                f"# Загружен YAML\n"
                f"- **topic**: {task1.get('topic')}\n"
                f"- **submission_id**: `{task1.get('submission_id')}`"
            ))

            bundle = build_task2_validation_bundle(
                trajectory_path,
                out_dir=Path(out_dir.value),
                include_auto_pipeline=bool(run_auto.value),
                multimodal=bool(multimodal.value),
                enable_reference_scout=bool(reference_scout.value),
            )
            manifest = _display_manifest(bundle.manifest_path)
            workspace = _build_review_workspace(manifest, task1)
            display(workspace)

            status.value = (
                '<span style="color:green"><b>Готово.</b> '
                'Откройте вкладки для просмотра assertions/графов и выполните разметку в разделе '
                '«Валидация эксперта».</span>'
            )
        except ValueError as e:
            status.value = f'<span style="color:#b45309"><b>Нужно действие:</b> {e}</span>'
            display(Markdown(
                '### Проверьте входной файл\n'
                '- загрузите один файл `.yaml` или `.yml`;\n'
                '- дождитесь зелёного индикатора **«YAML загружен»**;\n'
                '- проверьте, что рядом появился `submission_id` или размер файла;\n'
                '- убедитесь, что файл не пустой;\n'
                '- затем нажмите **«Построить Task 2 bundle»** ещё раз.'
            ))
        except Exception as e:
            status.value = f'<span style="color:red"><b>Ошибка:</b> {e}</span>'
            display(Markdown(
                '### Во время выполнения возникла ошибка\n'
                'Ноутбук не упал. Проверьте текст ошибки выше и исправьте входные данные или настройки.'
            ))


run_btn.on_click(_on_run)

display(W.VBox([
    W.HTML('<h2>Task 2 — Validation UI</h2>'),
    W.HTML(
        '<div style="color:#475569;margin-bottom:8px">'
        'Сначала загрузите YAML-файл из Task 1. После быстрой проверки рядом появится зелёный индикатор, '
        'а также `submission_id` или размер файла. После сборки bundle откроются вкладки просмотра графов '
        'и интерфейс экспертной валидации с экспортом готовых файлов.'
        '</div>'
    ),
    W.HBox([yaml_upload, yaml_indicator]),
    out_dir,
    run_auto,
    multimodal,
    reference_scout,
    run_btn,
    status,
    out,
]))


# CLI-режим (тот же workflow без notebook)

Если удобнее запускать всё одной командой из терминала, используйте:

```bash
top-papers-graph task2-bundle \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```

Эквивалентный основной алиас из репозитория:

```bash
top-papers-graph prepare-task2-validation \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```


# FAQ

## Что считается эталонным графом?
Эталонный граф строится **только из YAML Task 1** и полностью повторяет шаги эксперта, их evidence и связи между шагами.

## Что считается автоматическим графом?
Автоматический граф строится из **списка статей в YAML** и запускает встроенный конвейер репозитория: статьи → ingestion → temporal KG → review assets → report.

## Что делать, если автоматический граф почти пустой?
Это ожидаемо, если PDF недоступны, в статьях мало структурируемого текста или temporal evidence неявен. В этом случае ориентируйтесь на:
- `reference_scout.json`
- triplets CSV
- `comparison_summary.json`
- review templates из auto pipeline

## Как интерпретировать temporal поля?
- `start_date` / `end_date` — когда связь наблюдается / извлечена;
- `valid_from` / `valid_to` — когда открытие считается валидным;
- `temporal_basis` / `time_source` — откуда взялось время.
